# TDT4173 — SINTEF unit commitment: `exp_040_xgb_dispatch_capacity` (XGBoost)

**Kaggle team:** TODO "[<group number>] <team name>"

| Name | Student ID |
|---|---|
| TODO | TODO |

Reproduces prediction **exp_040_xgb_dispatch_capacity** (public LB not yet submitted, local CV 0.9927246 as the mean AUC over the leave-one-year-out folds 2019–2022).

Pipeline: load the raw data → physics features (water-value margins, scarcity, a per-plant dynamic program, Tokke–Vinje coupling) → **a relaxed LP of the whole watercourse per case** (our approximation of SINTEF's SHOP optimiser) → one pooled XGBoost model over all units and hours, trained on all 2015–2022 cases → `submission_exp_040_xgb_dispatch_capacity.csv`.

**How to run:** place the competition files next to this notebook (`data/kernel/`, `data/extended/`, `prediction_mapping.csv`, `sample_submission.csv`; or change `DATA_DIR` below), install `requirements.txt` (Python 3.12), then run all cells. CPU only, at most 4 threads; runtime and memory depend on the hardware. Training is deterministic: two runs give identical files. How the hyperparameters were chosen is described in the report.

In [ ]:
from pathlib import Path

DATA_DIR = Path(".")  # folder that holds data/, prediction_mapping.csv and sample_submission.csv
ROOT = DATA_DIR
KERNEL_DIR = DATA_DIR / "data" / "kernel"
EXTENDED_DIR = DATA_DIR / "data" / "extended"
OUTPUT = Path("submission_exp_040_xgb_dispatch_capacity.csv")

## 1. Data

### Loading and shared lookups

Raw data loading: train and test cases, input tables and topology.

Cases are indexed 0 … 3646 in Run No order: the 2922 train cases (2015–2022) first, then
the 725 test cases (2023-01-01 → 2024-12-25). Labels exist for the train cases only.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

HOURS = 168
LABEL_PREFIX = "result_committed_"

# Water routing per plant, from the `connections` in the topology YAML.
# Upstream: the reservoir(s) the plant takes its water from.
UPSTREAM = {
    "Hogga": ["Bandak"],
    "Tokke": ["Vinjevatn"],
    "Lio": ["Byrtevatn"],
    "Byrte": ["Botnedalsvatn"],
    "Vinje": ["Vaamarvatn"],
    "Haukeli": ["Vatjern"],
    "Songa": ["Songavatn", "Bitdalsvatn"],
    "Kjela": ["Foersvatn"],
    "Vesle Kjela": ["Kjelavatn"],
}
# Downstream: the reservoir the plant releases into (None = leaves the modelled system).
DOWNSTREAM = {
    "Hogga": None,
    "Tokke": "Bandak",
    "Lio": None,
    "Byrte": "Byrtevatn",
    "Vinje": "Vinjevatn",
    "Haukeli": "Vinjevatn",
    "Songa": "Totak",
    "Kjela": "Hyljelihyl",
    "Vesle Kjela": "Foersvatn",
}


@dataclass
class Data:
    run_no: np.ndarray  # (case,)
    starts: pd.Series  # (case,) case start times, UTC
    n_train: int  # cases 0 … n_train-1 have labels
    units: list[str]  # e.g. "Tokke_G1", in label-column order
    label_columns: list[str]  # submission column order (unit-major)
    Y: np.ndarray  # labels (train case, unit, hour)
    price: pd.Series  # day-ahead price, EUR/MWh (hourly up to 2025-09-30)
    inflow: pd.DataFrame  # hourly inflow per reservoir, m³/s
    volume: pd.DataFrame  # daily volume per reservoir, Mm³
    water_value: pd.DataFrame  # daily water value per reservoir, EUR/MWh
    max_vol: dict[str, float]  # reservoir -> max volume, Mm³
    generators: dict[str, dict]  # unit -> YAML generator entry (p_max, startcost_const, ...)

    @property
    def n_cases(self) -> int:
        return len(self.run_no)


def plant_of(unit: str) -> str:
    """'Vesle Kjela_G1' -> 'Vesle Kjela'."""
    return unit.rsplit("_", 1)[0]


def _read_timeseries(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, parse_dates=["date"]).set_index("date")


def load_data() -> Data:
    labels = pd.read_csv(KERNEL_DIR / "Unit_commitment_decisions.csv", parse_dates=["starttime"])
    test = pd.read_csv(ROOT / "prediction_mapping.csv", parse_dates=["starttime"])
    label_columns = [c for c in labels.columns if c.startswith(LABEL_PREFIX)]
    units = list(dict.fromkeys(c.removeprefix(LABEL_PREFIX).rsplit("_t", 1)[0] for c in label_columns))

    run_no = np.concatenate([labels["Run No"].to_numpy(), test["Run No"].to_numpy()])
    if not (np.diff(run_no) == 1).all():
        raise ValueError("Run No is not consecutive across train and test")
    topology = yaml.safe_load((EXTENDED_DIR / "Tokke_Vinje_topology.yaml").read_text())["model"]

    return Data(
        run_no=run_no,
        starts=pd.concat([labels["starttime"], test["starttime"]], ignore_index=True).dt.tz_localize("UTC"),
        n_train=len(labels),
        units=units,
        label_columns=label_columns,
        Y=labels[label_columns].to_numpy().reshape(len(labels), len(units), HOURS),
        price=_read_timeseries(KERNEL_DIR / "Historical_day_ahead_price_2015_2025.csv").iloc[:, 0],
        inflow=_read_timeseries(KERNEL_DIR / "Historical_inflow_1958_2025.csv"),
        volume=_read_timeseries(KERNEL_DIR / "Historical_volume_2015_2024.csv"),
        water_value=_read_timeseries(KERNEL_DIR / "Synthetic_water_value_2015_2024.csv"),
        max_vol={name: r["max_vol"] for name, r in topology["reservoir"].items()},
        generators=topology["generator"],
    )


# --- Lookups shared by the feature groups -------------------------------------------------

def hourly_window(series: pd.Series, starts: pd.Series, offset_hours: int = 0) -> np.ndarray:
    """The 168 hourly values from each start (+ offset), shape (case, hour).

    Assumes an hourly index without gaps, which holds for 2015–2024 (EDA §5.4).
    """
    first = series.index.get_indexer(starts + pd.Timedelta(hours=offset_hours))
    if (first < 0).any():
        raise KeyError("a case start is missing from the hourly index")
    return series.to_numpy()[first[:, None] + np.arange(HOURS)]


def daily_value(table: pd.DataFrame, days: pd.Series, columns: list[str]) -> np.ndarray:
    """Mean of `columns` on each day in `days`, shape (case,)."""
    return table.loc[days, columns].mean(axis=1).to_numpy()


def week_prices(data: Data, offset_hours: int = 0) -> np.ndarray:
    """Hourly prices of each case week (offset 168 = the week after), shape (case, hour)."""
    return hourly_window(data.price, data.starts, offset_hours)


def upstream_wv(data: Data, day_offset: int) -> np.ndarray:
    """Each unit's upstream water value on case start + `day_offset` days, shape (case, unit)."""
    days = data.starts + pd.Timedelta(days=day_offset)
    return np.stack([daily_value(data.water_value, days, UPSTREAM[plant_of(u)]) for u in data.units], axis=1)


def downstream_wv(data: Data, day_offset: int) -> np.ndarray:
    """Like upstream_wv for the downstream reservoir; 0 where the water leaves the system."""
    days = data.starts + pd.Timedelta(days=day_offset)
    columns = []
    for unit in data.units:
        down = DOWNSTREAM[plant_of(unit)]
        columns.append(daily_value(data.water_value, days, [down]) if down else np.zeros(data.n_cases))
    return np.stack(columns, axis=1)


## 2. Features

Each group builds arrays that broadcast to (case, unit, hour) from the inputs SHOP had for that case only: prices, inflows and constraints inside the week, the start-day volume, and the water values on the start day and start + 7 days. `load_group` builds a group once and keeps it in memory as float32, like the harness's feature cache (some groups reuse another group's output).

In [ ]:
_groups = {}


def load_group(group, data):
    """Features of one group, shape (case, unit, hour, feature), float32."""
    if group not in _groups:
        print(f"building feature group '{group}'", flush=True)
        features = globals()[f"build_{group}"](data)
        shape = (data.n_cases, len(data.units), HOURS)
        array = np.empty(shape + (len(features),), dtype=np.float32)
        for j, values in enumerate(features.values()):
            array[..., j] = np.broadcast_to(values, shape)
        _groups[group] = array, list(features)
    return _groups[group]

### Feature group `price_shape`

Price shape within the case week, shared by all units.

`price` and `price_wkmean` are raw EUR/MWh. The briefing (§2.4) prefers relative features;
they are kept here because baseline_v2 has them. Test dropping them as an experiment.

In [ ]:
import numpy as np
import pandas as pd



def build_price_shape(data):
    prices = week_prices(data)
    week_mean = prices.mean(axis=1, keepdims=True)
    week_std = prices.std(axis=1, keepdims=True) + 1e-3
    level = np.abs(week_mean) + 1
    rolling_24h = pd.DataFrame(prices.T).rolling(24, center=True, min_periods=1).mean().to_numpy().T

    hourly = {
        "price": prices,
        "price_z": (prices - week_mean) / week_std,
        "price_rank": prices.argsort(axis=1).argsort(axis=1) / (HOURS - 1),
        "price_rel": prices / level,
        "price_wkmean": np.repeat(week_mean, HOURS, axis=1),
        "price_d24": rolling_24h / level,
    }
    return {name: values[:, None, :] for name, values in hourly.items()}


### Feature group `calendar`

Calendar: hour within the case, hour of day (UTC), and the start day's position.

In [ ]:
import numpy as np



def build_calendar(data):
    hour = np.arange(HOURS)[None, None, :]
    return {
        "h": hour,
        "hod": hour % 24,
        "doy": data.starts.dt.dayofyear.to_numpy()[:, None, None],
        "dow": data.starts.dt.dayofweek.to_numpy()[:, None, None],
    }


### Feature group `water_value`

Water values per unit: upstream at start and +7 d, downstream at +7 d, and the net value.

In [ ]:
import numpy as np



def build_water_value(data):
    week_mean = week_prices(data).mean(axis=1)[:, None]
    up_end = upstream_wv(data, 7)
    up_start = upstream_wv(data, 0)
    down_end = downstream_wv(data, 7)
    net = up_end - down_end

    per_unit_week = {
        "wv_up_end": up_end,
        "wv_up_start": up_start,
        "wv_dn_end": down_end,
        "net_wv": net,
        "net_wv_rel": net / (np.abs(week_mean) + 1),
        "wv_trend": (up_end - up_start) / (np.abs(up_start) + 1),
    }
    return {name: values[:, :, None] for name, values in per_unit_week.items()}


### Feature group `hydro`

Reservoirs per unit: fill on the start day (upstream and downstream) and the week's inflow.

Only the start-day volume is used; later volumes are outcomes of the plan (leakage).

In [ ]:
import numpy as np



def build_hydro(data):
    starts = data.starts
    fill, fill_down, inflow = [], [], []
    for unit in data.units:
        up = UPSTREAM[plant_of(unit)]
        down = DOWNSTREAM[plant_of(unit)]
        capacity = sum(data.max_vol[r] for r in up)

        fill.append(sum(data.volume.loc[starts, r].to_numpy() for r in up) / capacity)
        fill_down.append(data.volume.loc[starts, down].to_numpy() / data.max_vol[down]
                         if down else np.zeros(data.n_cases))
        # Week inflow in Mm³ (m³/s × 3600 s / 1e6), relative to the upstream capacity
        week_inflow = sum(hourly_window(data.inflow[r], starts).sum(axis=1) * 3600 / 1e6 for r in up)
        inflow.append(week_inflow / capacity)

    return {
        "fill": np.stack(fill, axis=1)[:, :, None],
        "fill_dn": np.stack(fill_down, axis=1)[:, :, None],
        "inflow_wk": np.stack(inflow, axis=1)[:, :, None],
    }


### Feature group `unit_static`

Unit statics: unit ID (used as a categorical feature), p_max and start cost.

In [ ]:
import numpy as np


def build_unit_static(data):
    generators = [data.generators[u] for u in data.units]
    return {
        "unit": np.arange(len(data.units))[None, :, None],
        "p_max": np.array([g["p_max"] for g in generators])[None, :, None],
        "startcost": np.array([g["startcost_const"] for g in generators])[None, :, None],
    }


### Feature group `wv_margin`

Price vs water value per unit and hour: the core decision signal (BRIEFING §2.1, §3).

`margin_up` (price − upstream water value at +7 d) is the strong one. The net versions
subtract the downstream value too, which is much weaker on its own.

In [ ]:
import numpy as np



def build_wv_margin(data):
    prices = week_prices(data)[:, None, :]
    week_level = np.abs(prices.mean(axis=2, keepdims=True)) + 1
    scale = np.abs(prices).mean(axis=2, keepdims=True) + 1
    up_end = upstream_wv(data, 7)[:, :, None]
    net = up_end - downstream_wv(data, 7)[:, :, None]

    return {
        "margin": (prices - net) / week_level,
        "margin_up": (prices - up_end) / scale,
        "price_over_wv": prices / (np.abs(up_end) + 1),
        "margin_net": (prices - net) / scale,
    }


### Feature group `scarcity`

Scarcity coefficient c in water value ≈ c × mean price (BRIEFING §2.1), per unit.

Note: `scarcity_end` and `next_week_price` use the prices of the week *after* the case,
which is outside the §2.0 case inputs. It's not label leakage (the +7 d water value
already contains that week's mean price), but keep it in mind for the leakage audit.

In [ ]:
import numpy as np



def build_scarcity(data):
    week_mean = week_prices(data).mean(axis=1)[:, None]
    next_week_mean = week_prices(data, offset_hours=HOURS).mean(axis=1)[:, None]

    return {
        "scarcity_start": (upstream_wv(data, 0) / (np.abs(week_mean) + 1e-3))[:, :, None],
        "scarcity_end": (upstream_wv(data, 7) / (np.abs(next_week_mean) + 1e-3))[:, :, None],
        "next_week_price": (next_week_mean / (np.abs(week_mean) + 1))[:, :, None],
    }


### Feature group `case_economics`

Week-level economics per unit: how much of the week is worth running.

In [ ]:
import numpy as np



def build_case_economics(data):
    prices = week_prices(data)[:, None, :]
    scale = np.abs(prices).mean(axis=2, keepdims=True) + 1
    margin = prices - upstream_wv(data, 7)[:, :, None]

    return {
        "profitable_share": (margin > 0).mean(axis=2, keepdims=True),
        "positive_margin": np.clip(margin, 0, None).sum(axis=2, keepdims=True) / scale / HOURS,
    }


### Feature group `relaxed_dp`

Per-plant dynamic programme over the case week (BRIEFING §4.5, Level 1).

For each plant and case: the most profitable path of "number of units on" over the 168
hours, where k units earn k × p_max × (price − upstream water value at +7 d) per hour and
each start or stop costs the YAML start cost. Ignores reservoir limits (Level 2, the LP,
adds those).

Units within a plant follow a fixed priority (EDA §3.1.1, BRIEFING §2.3): the unit with
rank r is on when at least r units are on.

The start costs in the YAML may be on a different scale than SHOP effectively uses, so the
DP is solved twice: with the YAML cost (×1) and with ×10, which gives more persistence.

In [ ]:
import numpy as np


# Order in which a plant's units are committed (1 = first), from the label ON rates
PRIORITY = {"Tokke_G1": 1, "Tokke_G3": 2, "Tokke_G4": 3, "Tokke_G2": 4,
            "Vinje_G1": 1, "Vinje_G2": 2, "Vinje_G3": 3}
COST_SCALES = [1, 10]


def _path_values(reward: np.ndarray, switch_cost: np.ndarray) -> np.ndarray:
    """Best total profit of any path that is in state k at hour t, shape (case, hour, state).

    reward: (case, hour, state); switch_cost: (state from, state to).
    Forward pass (best way to arrive) + backward pass (best way to continue).
    """
    forward = np.empty_like(reward)
    backward = np.zeros_like(reward)
    forward[:, 0] = reward[:, 0]
    for t in range(1, HOURS):
        forward[:, t] = reward[:, t] + (forward[:, t - 1, :, None] - switch_cost).max(axis=1)
    for t in range(HOURS - 2, -1, -1):
        ahead = reward[:, t + 1] + backward[:, t + 1]
        backward[:, t] = (ahead[:, None, :] - switch_cost).max(axis=2)
    return forward + backward


def build_relaxed_dp(data):
    prices = week_prices(data)
    price_level = np.abs(prices).mean(axis=1, keepdims=True) + 1
    wv_up_end = upstream_wv(data, 7)
    n_cases, n_units = len(prices), len(data.units)

    features = {}
    for scale in COST_SCALES:
        on = np.zeros((n_cases, n_units, HOURS))
        gap = np.zeros((n_cases, n_units, HOURS))
        units_on = np.zeros((n_cases, n_units, HOURS))

        for plant in dict.fromkeys(plant_of(u) for u in data.units):
            members = [u for u, unit in enumerate(data.units) if plant_of(unit) == plant]
            first = data.generators[data.units[members[0]]]
            n_on = len(members)
            states = np.arange(n_on + 1)

            margin = prices - wv_up_end[:, members[0]][:, None]  # EUR/MWh, (case, hour)
            reward = margin[:, :, None] * states * first["p_max"]
            switch_cost = scale * first["startcost_const"] * np.abs(states[:, None] - states[None, :])
            values = _path_values(reward, switch_cost)

            best_state = values.argmax(axis=2)
            for u in members:
                rank = PRIORITY.get(data.units[u], 1)
                value_on = values[:, :, rank:].max(axis=2)
                value_off = values[:, :, :rank].max(axis=2)
                on[:, u] = best_state >= rank
                # Regret of the opposite decision, in hours of a typical margin for this unit
                gap[:, u] = (value_on - value_off) / (first["p_max"] * price_level)
                units_on[:, u] = best_state / n_on

        suffix = f"_x{scale}"
        features["dp_on" + suffix] = on
        features["dp_gap" + suffix] = np.clip(gap, -50, 50)
        features["dp_units" + suffix] = units_on
    return features


### Feature group `tokke_vinje`

Tokke–Vinje coupling (BRIEFING §2.2, EDA §4.4).

Vinje releases into Vinjevatn (9.4 Mm³, tiny), which Tokke empties in about 20 h, so the
two plants run together (label correlation 0.6–0.8). Each Tokke unit gets Vinje's DP
schedule and margin, and each Vinje unit gets Tokke's. Other units get NaN (LightGBM
treats it as missing).

Reads the relaxed_dp cache, so relaxed_dp must be buildable (it doesn't have to be in the
same config).

In [ ]:
import numpy as np


PARTNER = {"Tokke": "Vinje", "Vinje": "Tokke"}


def build_tokke_vinje(data):
    dp, dp_names = load_group("relaxed_dp", data)
    prices = week_prices(data)
    scale = np.abs(prices).mean(axis=1, keepdims=True) + 1
    margin = (prices[:, None, :] - upstream_wv(data, 7)[:, :, None]) / scale[:, None, :]

    first_unit = {}
    for u, unit in enumerate(data.units):
        first_unit.setdefault(plant_of(unit), u)

    shape = (data.n_cases, len(data.units), dp.shape[2])
    partner_units = np.full(shape, np.nan)
    partner_gap = np.full(shape, np.nan)
    partner_margin = np.full(shape, np.nan)
    for u, unit in enumerate(data.units):
        partner = PARTNER.get(plant_of(unit))
        if partner is None:
            continue
        p = first_unit[partner]  # plant-level values are the same for all its units
        partner_units[:, u] = dp[:, p, :, dp_names.index("dp_units_x1")]
        partner_gap[:, u] = dp[:, p, :, dp_names.index("dp_gap_x1")]
        partner_margin[:, u] = margin[:, p]

    return {
        "partner_dp_units": partner_units,
        "partner_dp_gap": partner_gap,
        "partner_margin": partner_margin,
    }


## 3. Relaxed LP of the watercourse

### LP model (relaxed_lp)

Relaxed LP of the whole watercourse (BRIEFING §4.5, Level 2: the main track).

One linear programme per case, built from the topology YAML: all plants, reservoirs,
tunnels and rivers, hourly over the 168 hours. On/off status is relaxed: a plant may run
at any level between 0 and full capacity. The LP maximises

    income from selling power + value of the water left in the reservoirs after 168 h

subject to the water balance of every reservoir in every hour. Its solution (how much
each plant produces, how full each reservoir gets) and its duals (the effective hourly
value of water) become features; LightGBM learns the corrections.

Default model (each refinement below is an option, so earlier feature groups stay
reproducible):
- Production = k × discharge, k fixed per case from the start-day head
  (k = 9.81e-3 × head × best efficiency, MW per m³/s). No start costs, no p_min.
- Songavatn and Bitdalsvatn are merged (same levels and water value).
- Tunnels move water freely in both directions (they join lakes at the same level).
- River releases (spill, bypass) cost a tiny penalty; every reservoir also has an
  expensive emergency overflow, so no LP is infeasible.
- Min-volume constraints are lower bounds on the volume, capped at the start volume.
- End value: water left after 168 h is valued leg by leg down the cascade, each plant's
  production at its own upstream water value (d+7); see _end_values.

Options (solve_all / solve_case keyword arguments):
- min_flow: the six min-flow constraints, soft (shortfall costs MIN_FLOW_PENALTY).
- commitment: relaxed unit commitment (units on, p_min, start/stop costs).
- production_curves: production as a concave piecewise-linear function of discharge,
  from the turbine efficiency curves, generator efficiency and head losses in the YAML
  (instead of one constant k at the best efficiency).
- flow_costs="yaml": the rivers' flow_cost_const from the YAML (spill 1000, bypass 10 per
  m³/s·h) instead of the tiny penalty, so spilling is a last resort as in SHOP.
  flow_costs="yaml_curves": the same, plus the four main rivers that have a
  flow_cost_curve instead (r_Staavatn, r_Langesae, r_Langeidvatn, r_Hyljelihyl; cost 0
  with "yaml") at the curve's average cost, y / x at its last point (≈ 1.76 per m³/s·h).
  Without it, moving water down these rivers is free, so the LP shifts it arbitrarily
  (e.g. Langesae into Foersvatn in the last hours).
- terminal_efficiency: "full" (default) converts terminal water to energy at full-load
  average efficiency; "best" uses the first, most efficient production segment.
  This changes the terminal objective only, leaving in-horizon production unchanged.
- head_iterations: re-solve with the head at each reservoir's mean level in the previous
  solution (SHOP's successive linearisation, in its simplest form).
- integer_units (needs commitment): after the LP, solve the same problem as a MILP with
  whole numbers of units on (BRIEFING §4.5 Level 3), with a time limit per case.
  production / units_on / fill come from the MILP; margin and water_value stay the LP's
  duals (a MILP has none).
- main_river (solve_all only; it changes the network): a reservoir's end value follows
  the river marked `main_river` in the YAML instead of the first river found. Matters
  for Hyljelihyl only: r_Hyljelihyl → Venemo → Totak → Vinje …, not the bypass
  b_Hyljelihyl → Vinjevatn that skips Vinje.

Uses only the §2.0 case inputs: prices and inflows over the 168 h, start-day volumes,
the +7 d water values and the constraint tables.

In [ ]:
import ast
import os
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from functools import lru_cache

import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml
from scipy.optimize import Bounds, LinearConstraint, linprog, milp


MERGED = {"Bitdalsvatn": "Songavatn"}  # reservoir -> the reservoir it's merged into
MM3_PER_M3S_HOUR = 0.0036  # 1 m³/s for one hour = 0.0036 Mm³
RELEASE_PENALTY = 0.01  # EUR per m³/s·h for spill and bypass (default flow costs)
OVERFLOW_PENALTY = 1000.0  # EUR per m³/s·h for the emergency overflow
MIN_FLOW_PENALTY = 1000.0  # EUR per m³/s·h of shortfall below a min-flow constraint
TUNNEL_CAPACITY = 1000.0  # m³/s, effectively unlimited
GENERATOR_EFFICIENCY = 0.98  # default model only; production_curves uses the YAML curve
CURVE_SEGMENTS = 4  # linear pieces per plant with production_curves
MIN_FLOW_RIVERS = ["b_Kjelavatn", "b_Hyljelihyl", "b_Vinjevatn", "b_Byrtevatn", "b_Bandak", "r_Vest_Vassdraget"]
END_VALUE_BASIS = "path"  # how leftover water is valued; see _end_values
WORKERS = 1  # in-process: worker processes can't import functions defined in a notebook
PRIORITY = {"Tokke_G1": 1, "Tokke_G3": 2, "Tokke_G4": 3, "Tokke_G2": 4,
            "Vinje_G1": 1, "Vinje_G2": 2, "Vinje_G3": 3}  # as in relaxed_dp


@dataclass
class Edge:
    """One water flow (m³/s), from a reservoir to a reservoir; None = outside the system."""
    name: str
    kind: str  # "plant", "tunnel", "river" or "overflow"
    source: int | None
    target: int | None
    upper: float  # m³/s; plant segments get their width per case
    lower: float = 0.0
    via: tuple[str, ...] = ()  # rivers the water passes on its way to the target
    flow_cost: float = 0.0  # the YAML's flow_cost_const (rivers)
    curve_cost: float = 0.0  # average cost of the YAML's flow_cost_curve, if the river has one
    plant: int | None = None  # for plant segments: index into Network.plants


@dataclass
class Plant:
    """What the production function needs, from the YAML."""
    units: int
    pmin: float  # per unit, MW
    pmax: float
    start_cost: float  # EUR per start of one unit
    stop_cost: float
    outlet: float  # outlet level, m
    best_efficiency: float  # turbine × generator at the best point (default model)
    turbine_curves: list[dict]  # [{ref: head, x: unit flow, y: efficiency %}, ...]
    generator_curve: dict  # {x: MW, y: efficiency %}
    main_loss: float  # head loss = main_loss × Q² (plant flow) + penstock_loss × q² (penstock flow)
    penstock_loss: float
    penstocks: int


@dataclass
class Network:
    reservoirs: list[str]
    max_vol: np.ndarray  # (reservoir,)
    level_curves: list[tuple[list, list]]  # volume (Mm³) -> level (m), per reservoir
    plants: list[str]
    plant_data: list[Plant]
    edges: list[Edge]  # plant segments first (plant-major), then tunnels, rivers, overflows
    segments: int  # linear pieces per plant
    main_outlet: list[int | None]  # per reservoir: the edge its water mainly leaves by
    A_eq: sp.csr_matrix  # water balance, rows (hour, reservoir)
    plant_edges: np.ndarray = field(init=False)  # (plant, segment) -> edge index

    def __post_init__(self):
        self.plant_edges = np.arange(len(self.plants) * self.segments).reshape(len(self.plants), self.segments)

    def plant_source(self, p: int) -> int:
        return self.edges[self.plant_edges[p, 0]].source

    def plant_target(self, p: int) -> int | None:
        return self.edges[self.plant_edges[p, 0]].target


def _merged(name: str) -> str:
    return MERGED.get(name, name)


@lru_cache(maxsize=4)
def build_network(segments: int = 1, main_river: bool = False) -> Network:
    topology = yaml.safe_load((EXTENDED_DIR / "Tokke_Vinje_topology.yaml").read_text())
    model, connections = topology["model"], topology["connections"]
    targets = {}
    for c in connections:
        targets.setdefault((c["from_type"], c["from"]), []).append((c["to_type"], c["to"]))

    def route(kind: str, name: str, via: tuple = ()) -> tuple[str | None, tuple]:
        """Follow rivers downstream until a reservoir (None = leaves the system); also
        returns the rivers passed on the way."""
        for to_type, to in targets.get((kind, name), []):
            if to_type == "reservoir":
                return _merged(to), via
            if to_type == "river":
                return route("river", to, via + (to,))
        return None, via

    reservoirs = [r for r in model["reservoir"] if r not in MERGED]
    index = {r: i for i, r in enumerate(reservoirs)}
    max_vol = np.zeros(len(reservoirs))
    for r, spec in model["reservoir"].items():
        max_vol[index[_merged(r)]] += spec["max_vol"]
    level_curves = [(model["reservoir"][r]["vol_head"]["x"], model["reservoir"][r]["vol_head"]["y"])
                    for r in reservoirs]
    for r, into in MERGED.items():  # merged lake: scale the main lake's curve to the joint volume
        x, y = level_curves[index[into]]
        scale = max_vol[index[into]] / model["reservoir"][into]["max_vol"]
        level_curves[index[into]] = ([v * scale for v in x], y)

    sources_of = {}  # plant or tunnel -> reservoir that feeds it
    for c in connections:
        if c["from_type"] == "reservoir" and c["to_type"] in ("plant", "tunnel"):
            sources_of[c["to"]] = _merged(c["from"])

    edges, plants, plant_data = [], [], []
    for p, (plant, spec) in enumerate(model["plant"].items()):
        source = sources_of.get(plant)
        if source is None:  # fed through tunnels (Songa)
            source = next(sources_of[t] for t in model["tunnel"] if ("plant", plant) in targets.get(("tunnel", t), []))
        units = [g for g in model["generator"] if plant_of(g) == plant]
        first = model["generator"][units[0]]
        curves = first["turb_eff_curves"]
        curves = ast.literal_eval(curves) if isinstance(curves, str) else curves
        target, via = route("plant", plant)
        for s in range(segments):
            edges.append(Edge(f"{plant}#{s}", "plant", index[source], index[target] if target else None,
                              upper=np.inf, via=via, plant=p))
        plants.append(plant)
        plant_data.append(Plant(
            units=len(units), pmin=first["p_min"], pmax=first["p_max"],
            start_cost=first["startcost_const"], stop_cost=first.get("stopcost_const", 0.0),
            outlet=spec["outlet_line"],
            best_efficiency=max(max(c["y"]) for c in curves) / 100 * GENERATOR_EFFICIENCY,
            turbine_curves=curves, generator_curve=first["gen_eff_curve"],
            main_loss=float(spec["main_loss"][0]), penstock_loss=float(spec["penstock_loss"][0]),
            penstocks=len({model["generator"][g]["penstock"] for g in units}),
        ))

    for tunnel in model["tunnel"]:
        target = route("tunnel", tunnel)[0]
        if target is not None:  # tunnels into plants are part of the plant edge
            edges.append(Edge(tunnel, "tunnel", index[sources_of[tunnel]], index[target],
                              upper=TUNNEL_CAPACITY, lower=-TUNNEL_CAPACITY))

    for c in connections:
        if c["from_type"] == "reservoir" and c["to_type"] == "river":
            source, (target, via) = index[_merged(c["from"])], route("river", c["to"])
            target = index[target] if target else None
            if source != target:
                river = model["river"].get(c["to"], {})
                curve = river.get("flow_cost_curve")
                edges.append(Edge(c["to"], "river", source, target, upper=river.get("max_flow_const", np.inf),
                                  via=(c["to"],) + via, flow_cost=river.get("flow_cost_const", 0.0),
                                  curve_cost=curve["y"][-1] / curve["x"][-1] if curve else 0.0))
    for r in reservoirs:
        edges.append(Edge(f"overflow_{r}", "overflow", index[r], None, upper=np.inf))

    def main_outlet(r: int) -> int | None:
        """Plant if the reservoir feeds one, else a tunnel, else the first river onwards.
        With main_river, a river marked `main_river` in the YAML wins over the others
        (Hyljelihyl: r_Hyljelihyl → Venemo, not the bypass b_Hyljelihyl → Vinjevatn)."""
        for kind in ("plant", "tunnel", "river"):
            candidates = [e for e, edge in enumerate(edges)
                          if edge.kind == kind and edge.source == r and (kind == "plant" or edge.target is not None)]
            if main_river and kind == "river":
                candidates.sort(key=lambda e: not model["river"].get(edges[e].name, {}).get("main_river"))
            if candidates:
                return candidates[0]
        return None

    incidence = np.zeros((len(reservoirs), len(edges)))
    for e, edge in enumerate(edges):
        if edge.source is not None:
            incidence[edge.source, e] -= 1
        if edge.target is not None:
            incidence[edge.target, e] += 1
    # V[t] - V[t-1] - 0.0036 × (inflows - outflows)[t] = 0.0036 × natural inflow[t]
    step = sp.eye(HOURS) - sp.eye(HOURS, k=-1)
    A_eq = sp.hstack([-MM3_PER_M3S_HOUR * sp.kron(sp.eye(HOURS), sp.csr_matrix(incidence)),
                      sp.kron(step, sp.eye(len(reservoirs)))]).tocsr()

    return Network(reservoirs, max_vol, level_curves, plants, plant_data, edges, segments,
                   [main_outlet(r) for r in range(len(reservoirs))], A_eq)


def _production_function(plant: Plant, head: float, curves: bool, segments: int) -> tuple[np.ndarray, np.ndarray]:
    """Slopes k (MW per m³/s) and widths (m³/s) of the plant's linear pieces, best first.

    Default: one piece, k at the best efficiency, up to full capacity. With curves: the
    upper concave envelope of the plant's production over discharge, for 1 … n units
    sharing the flow equally, from the turbine curve closest to this head, the generator
    curve and the head losses; split into `segments` pieces of equal width.
    """
    if not curves:
        k = 9.81e-3 * head * plant.best_efficiency
        return np.array([k]), np.array([plant.units * plant.pmax / k])

    curve = min(plant.turbine_curves, key=lambda c: abs(c["ref"] - head))
    unit_flow, efficiency = np.asarray(curve["x"], float), np.asarray(curve["y"], float) / 100
    points = [(0.0, 0.0)]
    for m in range(1, plant.units + 1):
        flow = m * unit_flow
        penstock_flow = flow / min(m, plant.penstocks)
        net_head = np.maximum(head - plant.main_loss * flow ** 2 - plant.penstock_loss * penstock_flow ** 2, 0)
        mw = 9.81e-3 * net_head * flow * efficiency
        mw *= np.interp(mw / m, plant.generator_curve["x"], plant.generator_curve["y"]) / 100
        points += list(zip(flow, np.minimum(mw, m * plant.pmax)))
    points = sorted(points)

    hull = [points[0]]  # upper concave envelope (monotone chain)
    for q, mw in points[1:]:
        while len(hull) >= 2 and (hull[-1][1] - hull[-2][1]) * (q - hull[-2][0]) <= (mw - hull[-2][1]) * (hull[-1][0] - hull[-2][0]):
            hull.pop()
        hull.append((q, mw))
    hull_q, hull_mw = np.array(hull).T
    top = int(np.argmax(hull_mw))  # beyond the maximum, more flow doesn't add power
    hull_q, hull_mw = hull_q[:top + 1], hull_mw[:top + 1]

    breaks = np.linspace(0, hull_q[-1], segments + 1)
    mw_at = np.interp(breaks, hull_q, hull_mw)
    widths = np.diff(breaks)
    return np.maximum(np.diff(mw_at) / widths, 0), widths


def _end_values(net: Network, k_avg: np.ndarray, water_value: np.ndarray, basis: str,
                wv_from: dict[str, str] | None = None) -> np.ndarray:
    """Value (EUR) of 1 Mm³ left in each reservoir after 168 h, (reservoir,).

    Water left in a reservoir would later run through the plants on its way down
    (its main outlet, then the next reservoir's, ...). Each plant on the way turns
    1 Mm³ into k / 0.0036 MWh (k_avg: the plant's average MW per m³/s at full load).
    The bases differ in which water value prices that energy:
    - "path": each plant's MWh at the water value of the reservoir feeding that plant.
      Then every plant's trade-off is price vs its own upstream water value, which is
      what the labels follow (BRIEFING §3); the default.
    - "sea": all MWh down to the sea at the reservoir's own water value (fails the
      sanity check: the synthetic water values of neighbouring reservoirs aren't
      consistent with each other).
    - "local": only the first plant's MWh, at the reservoir's own water value.
    wv_from overrides, for "path", which reservoir's water value prices a plant's leg,
    e.g. {"Vinje": "Vinjevatn"}.
    """
    values = {}

    def walk(r: int, seen: frozenset) -> float:
        if r in values:
            return values[r]
        e = net.main_outlet[r]
        if e is None or r in seen:
            return 0.0
        edge = net.edges[e]
        mwh = k_avg[edge.plant] / MM3_PER_M3S_HOUR if edge.kind == "plant" else 0.0
        below = walk(edge.target, seen | {r}) if edge.target is not None else 0.0
        if basis == "path":
            plant_name = net.plants[edge.plant] if edge.kind == "plant" else None
            priced_at = net.reservoirs.index(wv_from[plant_name]) if plant_name in (wv_from or {}) else r
            values[r] = mwh * water_value[priced_at] + below
        elif basis == "local":
            values[r] = mwh * water_value[r] if edge.kind == "plant" else below * water_value[r] / max(
                water_value[edge.target], 1e-9) if edge.target is not None else 0.0
        else:  # sea: everything below priced at this reservoir's own water value
            values[r] = (mwh + (below / max(water_value[edge.target], 1e-9) if edge.target is not None else 0.0)) * water_value[r]
        return values[r]

    return np.array([walk(r, frozenset()) for r in range(len(net.reservoirs))])


def solve_case(net: Network, prices: np.ndarray, inflow: np.ndarray, start_volume: np.ndarray,
               end_water_value: np.ndarray, min_volume: np.ndarray,
               required_flow: np.ndarray | None = None, basis: str = END_VALUE_BASIS,
               wv_from: dict[str, str] | None = None, commitment: bool = False,
               production_curves: bool = False, flow_costs: str = "penalty",
               head_iterations: int = 0, integer_units: bool = False,
               milp_time_limit: float = 10.0, terminal_efficiency: str = "full") -> dict[str, np.ndarray] | None:
    """Solve one case. prices (hour,), inflow and min_volume (hour, reservoir) in m³/s and
    Mm³, start_volume and end_water_value (reservoir,), required_flow (hour, MIN_FLOW
    column) in m³/s or None for no min-flow constraints. Options: see the module
    docstring. Returns per-plant arrays of shape (hour, plant)."""
    if terminal_efficiency not in ("full", "best"):
        raise ValueError("terminal_efficiency must be full or best")
    levels = np.array([np.interp(v, *curve) for v, curve in zip(start_volume, net.level_curves)])
    result = None
    for _ in range(head_iterations + 1):
        result = _solve_once(net, levels, prices, inflow, start_volume, end_water_value, min_volume,
                             required_flow, basis, wv_from, commitment, production_curves, flow_costs,
                             integer_units, milp_time_limit, terminal_efficiency)
        if result is None:
            return None
        mean_volume = result["_volumes"].mean(axis=0)  # kept in the result for solve_all(volumes=True)
        levels = np.array([np.interp(v, *curve) for v, curve in zip(mean_volume, net.level_curves)])
    return result


def _solve_once(net, levels, prices, inflow, start_volume, end_water_value, min_volume,
                required_flow, basis, wv_from, commitment, production_curves, flow_costs,
                integer_units=False, milp_time_limit=10.0, terminal_efficiency="full"):
    n_res, n_edges, n_plants = len(net.reservoirs), len(net.edges), len(net.plants)
    heads = np.array([max(levels[net.plant_source(p)] - net.plant_data[p].outlet, 1.0) for p in range(n_plants)])
    pieces = [_production_function(net.plant_data[p], heads[p], production_curves, net.segments) for p in range(n_plants)]
    k = np.zeros(n_edges)  # MW per m³/s for plant segments
    lower = np.array([e.lower for e in net.edges], dtype=float)
    upper = np.array([e.upper for e in net.edges], dtype=float)
    for p, (slopes, widths) in enumerate(pieces):
        k[net.plant_edges[p]] = slopes
        upper[net.plant_edges[p]] = widths
    capacity_flow = np.array([w.sum() for _, w in pieces])
    capacity_mw = np.array([(s * w).sum() for s, w in pieces])
    k_avg = capacity_mw / capacity_flow  # MW per m³/s at full load

    if flow_costs in ("yaml", "yaml_curves"):
        with_curves = flow_costs == "yaml_curves"
        cost = np.array([e.flow_cost + with_curves * e.curve_cost if e.kind == "river"
                         else OVERFLOW_PENALTY if e.kind == "overflow" else 0.0 for e in net.edges])
    else:
        cost = np.array([{"river": RELEASE_PENALTY, "overflow": OVERFLOW_PENALTY}.get(e.kind, 0.0) for e in net.edges])
    flow_cost = np.tile(cost, (HOURS, 1)) - prices[:, None] * k[None, :]  # income from production
    volume_cost = np.zeros((HOURS, n_res))
    terminal_k = k_avg if terminal_efficiency == "full" else np.array([s[0] for s, w in pieces])
    volume_cost[-1] = -_end_values(net, terminal_k, end_water_value, basis, wv_from)  # EUR per Mm³ left at the end
    vol_upper = np.maximum(net.max_vol, start_volume)
    vol_lower = np.minimum(min_volume, start_volume)  # (hour, reservoir); feasible by construction

    b_eq = MM3_PER_M3S_HOUR * inflow
    b_eq[0] += start_volume
    bounds = np.vstack([np.column_stack([np.tile(lower, HOURS), np.tile(upper, HOURS)]),
                        np.column_stack([vol_lower.ravel(), np.tile(vol_upper, HOURS)])])

    # Variable blocks after the flows and volumes are added below; each block has a size,
    # a cost, bounds, and its columns in the inequality rows (A_ub x ≤ b_ub).
    sizes = {"flow": HOURS * n_edges, "volume": HOURS * n_res}
    costs, bound_parts = [flow_cost.ravel(), volume_cost.ravel()], [bounds]
    ub_blocks, ub_rhs = [], []

    def add_variables(name, size, cost, low, high):
        sizes[name] = size
        costs.append(np.broadcast_to(cost, size).astype(float))
        bound_parts.append(np.column_stack([np.broadcast_to(low, size), np.broadcast_to(high, size)]))

    if required_flow is not None:
        # Shortfall z ≥ 0 per constraint and hour: −(flows through the river) − z ≤ −required
        n_c = required_flow.shape[1]
        through = np.zeros((n_c, n_edges))
        for c, river in enumerate(MIN_FLOW_RIVERS):
            through[c, [e for e, edge in enumerate(net.edges) if river in edge.via]] = -1
        add_variables("shortfall", HOURS * n_c, MIN_FLOW_PENALTY, 0, np.inf)
        ub_blocks.append({"flow": sp.kron(sp.eye(HOURS), sp.csr_matrix(through)), "shortfall": -sp.eye(HOURS * n_c)})
        ub_rhs.append(-required_flow.ravel())

    if commitment:
        # Relaxed unit commitment per plant and hour: "on" units in [0, n_units] (fractional),
        # production between on × p_min and on × p_max, start/stop costs on changes in "on".
        # The state before the case is unknown, so hour 0 starts free.
        plant_units = np.array([d.units for d in net.plant_data], float)
        add_variables("on", HOURS * n_plants, 0, 0, np.tile(plant_units, HOURS))
        add_variables("start", HOURS * n_plants, np.tile([d.start_cost for d in net.plant_data], HOURS), 0, np.inf)
        add_variables("stop", HOURS * n_plants, np.tile([d.stop_cost for d in net.plant_data], HOURS), 0, np.inf)
        hours, plants = sp.eye(HOURS), sp.eye(n_plants)
        mw_of_plant = np.zeros((n_plants, n_edges))
        for p in range(n_plants):
            mw_of_plant[p, net.plant_edges[p]] = k[net.plant_edges[p]]
        production = sp.kron(hours, sp.csr_matrix(mw_of_plant))  # MW per plant and hour
        change = sp.kron((sp.eye(HOURS) - sp.eye(HOURS, k=-1)).tocsr()[1:], plants)  # on[t] − on[t−1]
        later = sp.kron(sp.eye(HOURS).tocsr()[1:], plants)
        ub_blocks += [
            {"flow": production, "on": -sp.kron(hours, sp.diags([d.pmax for d in net.plant_data]))},  # MW ≤ on × p_max
            {"flow": -production, "on": sp.kron(hours, sp.diags([d.pmin for d in net.plant_data]))},  # MW ≥ on × p_min
            {"on": change, "start": -later},                                                          # start ≥ increase
            {"on": -change, "stop": -later},                                                          # stop ≥ decrease
        ]
        ub_rhs += [np.zeros(HOURS * n_plants)] * 2 + [np.zeros((HOURS - 1) * n_plants)] * 2

    def row_block(columns: dict) -> sp.csr_matrix:
        n_rows = next(iter(columns.values())).shape[0]
        return sp.hstack([columns.get(name, sp.csr_matrix((n_rows, size))) for name, size in sizes.items()])

    n_extra = sum(sizes.values()) - HOURS * (n_edges + n_res)
    A_eq = sp.hstack([net.A_eq, sp.csr_matrix((net.A_eq.shape[0], n_extra))]).tocsr() if n_extra else net.A_eq
    A_ub = sp.vstack([row_block(b) for b in ub_blocks]).tocsr() if ub_blocks else None
    b_ub = np.concatenate(ub_rhs) if ub_rhs else None
    res = linprog(np.concatenate(costs), A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq.ravel(),
                  bounds=np.vstack(bound_parts), method="highs")
    if res.status != 0:
        return None

    flows = res.x[:HOURS * n_edges].reshape(HOURS, n_edges)
    volumes = res.x[HOURS * n_edges:HOURS * (n_edges + n_res)].reshape(HOURS, n_res)
    plant_mw = np.stack([(flows[:, net.plant_edges[p]] * k[net.plant_edges[p]]).sum(axis=1) for p in range(n_plants)], axis=1)
    # Duals: change in the (minimised) objective per extra Mm³ in a reservoir at hour t,
    # so the value of water there is −dual. Convert to EUR per MWh of this plant.
    water_value = -res.eqlin.marginals.reshape(HOURS, n_res)
    to_mwh = MM3_PER_M3S_HOUR / k_avg
    sources = np.array([net.plant_source(p) for p in range(n_plants)])
    targets = [net.plant_target(p) for p in range(n_plants)]
    wv_source = water_value[:, sources] * to_mwh
    wv_target = np.column_stack([water_value[:, t] if t is not None else np.zeros(HOURS) for t in targets]) * to_mwh
    result = {
        "production": plant_mw / capacity_mw,  # share of plant capacity
        "margin": prices[:, None] - wv_source + wv_target,  # EUR/MWh: profit of one more MWh
        "water_value": wv_source,
        "fill": volumes[:, sources] / net.max_vol[sources],
        "_volumes": volumes,
    }
    on_offset = sum(size for name, size in sizes.items() if name in ("flow", "volume", "shortfall"))
    if commitment:
        result["units_on"] = res.x[on_offset:on_offset + HOURS * n_plants].reshape(HOURS, n_plants)
    if integer_units and commitment:
        integrality = np.zeros(len(res.x))
        integrality[on_offset:on_offset + HOURS * n_plants] = 1
        all_bounds = np.vstack(bound_parts)
        constraints = [LinearConstraint(A_eq, b_eq.ravel(), b_eq.ravel())]
        if A_ub is not None:
            constraints.append(LinearConstraint(A_ub, -np.inf, b_ub))
        mip = milp(np.concatenate(costs), integrality=integrality, bounds=Bounds(all_bounds[:, 0], all_bounds[:, 1]),
                   constraints=constraints, options={"time_limit": milp_time_limit, "mip_rel_gap": 1e-3})
        if mip.x is not None:  # best solution found within the time limit
            flows = mip.x[:HOURS * n_edges].reshape(HOURS, n_edges)
            volumes = mip.x[HOURS * n_edges:HOURS * (n_edges + n_res)].reshape(HOURS, n_res)
            plant_mw = np.stack([(flows[:, net.plant_edges[p]] * k[net.plant_edges[p]]).sum(axis=1)
                                 for p in range(n_plants)], axis=1)
            result.update(production=plant_mw / capacity_mw, fill=volumes[:, sources] / net.max_vol[sources],
                          units_on=mip.x[on_offset:on_offset + HOURS * n_plants].reshape(HOURS, n_plants),
                          _volumes=volumes)
    return result


def _solve_chunk(args):
    options, inputs = args  # inputs: case_inputs' arrays, sliced to a chunk
    options = dict(options)
    net = build_network(CURVE_SEGMENTS if options.get("production_curves") else 1, options.pop("main_river", False))
    return [solve_case(net, *case, **options) for case in zip(*inputs)]


def case_inputs(data, net: Network, case_idx: np.ndarray):
    """The §2.0 inputs per case: prices, inflows per (merged) reservoir, start volumes,
    +7 d water values, min-volume constraints and the flow required by the min-flow
    constraints."""
    starts = data.starts.iloc[case_idx].reset_index(drop=True)
    prices = week_prices(data)[case_idx]
    index = {r: i for i, r in enumerate(net.reservoirs)}

    topology = yaml.safe_load((EXTENDED_DIR / "Tokke_Vinje_topology.yaml").read_text())
    river_target = {c["from"]: c["to"] for c in topology["connections"] if c["from_type"] == "river"}
    inflow = np.zeros((len(case_idx), HOURS, len(net.reservoirs)))
    for column in data.inflow.columns:
        target = column if column in data.max_vol else river_target[column]  # e.g. r_Vest_Vassdraget → Bandak
        inflow[:, :, index[_merged(target)]] += hourly_window(data.inflow[column], starts)

    start_volume = np.zeros((len(case_idx), len(net.reservoirs)))
    for r in data.max_vol:
        start_volume[:, index[_merged(r)]] += data.volume.loc[starts, r].to_numpy()
    end_days = starts + pd.Timedelta(days=7)
    end_wv = np.column_stack([data.water_value.loc[end_days, r].to_numpy() for r in net.reservoirs])
    constraint = pd.read_csv(EXTENDED_DIR / "Constraint_min_volume.csv", parse_dates=["date"]).set_index("date")
    min_volume = np.zeros((len(case_idx), HOURS, len(net.reservoirs)))
    for r in constraint.columns:
        min_volume[:, :, index[_merged(r)]] = hourly_window(constraint[r], starts)
    flows = pd.read_csv(EXTENDED_DIR / "Constraint_min_flow.csv", parse_dates=["date"]).set_index("date")
    required = np.stack([hourly_window(flows[river], starts) for river in MIN_FLOW_RIVERS], axis=2)
    # r_Vest_Vassdraget also carries its own natural inflow, which counts towards its minimum
    natural = np.stack([hourly_window(data.inflow[river], starts) if river in data.inflow else np.zeros((len(starts), HOURS))
                        for river in MIN_FLOW_RIVERS], axis=2)
    required_flow = np.maximum(required - natural, 0)
    return prices, inflow, start_volume, end_wv, min_volume, required_flow


def solve_all(data, case_idx: np.ndarray, workers: int | None = None, min_flow: bool = False,
              volumes: bool = False, **options) -> dict[str, np.ndarray]:
    """Solve the given cases in parallel. Returns arrays (case, hour, plant); NaN if an LP
    failed. `options` are solve_case's keyword arguments (basis, commitment, ...) and
    main_river. With `volumes=True` it also returns the LP's volume trajectories
    (case, hour, reservoir) as "volumes", with the network's "reservoirs" and "max_vol"."""
    net = build_network(CURVE_SEGMENTS if options.get("production_curves") else 1, options.get("main_river", False))
    inputs = case_inputs(data, net, case_idx)
    if not min_flow:
        inputs = inputs[:-1]  # drop required_flow
    workers = workers or WORKERS or max(1, (os.cpu_count() or 2) - 1)
    chunks = np.array_split(np.arange(len(case_idx)), workers * 4)
    jobs = [(options, [a[c] for a in inputs]) for c in chunks]
    if workers == 1:  # in-process: worker processes can't import functions defined in a notebook
        results = [r for job in jobs for r in _solve_chunk(job)]
    else:
        with ProcessPoolExecutor(workers) as pool:
            results = [r for chunk in pool.map(_solve_chunk, jobs) for r in chunk]

    failed = sum(r is None for r in results)
    if failed:
        print(f"    {failed} of {len(results)} LPs failed; their features are NaN")
    keys = ["production", "margin", "water_value", "fill"] + (["units_on"] if options.get("commitment") else [])
    empty = np.full((HOURS, len(net.plants)), np.nan)
    out = {key: np.stack([r[key] if r is not None else empty for r in results]) for key in keys} | {
        "plants": net.plants}
    if volumes:
        no_volumes = np.full((HOURS, len(net.reservoirs)), np.nan)
        out |= {"volumes": np.stack([r["_volumes"] if r is not None else no_volumes for r in results]),
                "reservoirs": net.reservoirs, "max_vol": net.max_vol}
    return out


def build_relaxed_lp(data):
    return lp_features(data, solve_all(data, np.arange(data.n_cases)), prefix="lp")


def lp_features(data, lp: dict, prefix: str) -> dict[str, np.ndarray]:
    """Per-unit features from solve_all's per-plant results."""
    plant_index = {p: i for i, p in enumerate(lp["plants"])}
    scale = np.abs(week_prices(data)).mean(axis=1, keepdims=True) + 1  # (case, 1)
    n_units = {p: sum(plant_of(u) == p for u in data.units) for p in plant_index}

    features = {name: np.zeros((data.n_cases, len(data.units), HOURS))
                for name in ["production", "unit_on", "margin", "water_value", "fill"]}
    for u, unit in enumerate(data.units):
        p = plant_index[plant_of(unit)]
        production = lp["production"][:, :, p]
        features["production"][:, u] = production
        # Units are committed in priority order: the r-th unit runs once r-1 are fully on.
        # With commitment the LP has its own units-on; otherwise derive it from production.
        units_on = lp["units_on"][:, :, p] if "units_on" in lp else production * n_units[plant_of(unit)]
        rank = PRIORITY.get(unit, 1)
        features["unit_on"][:, u] = np.clip(units_on - (rank - 1), 0, 1)
        features["margin"][:, u] = lp["margin"][:, :, p] / scale
        features["water_value"][:, u] = lp["water_value"][:, :, p] / scale
        features["fill"][:, u] = lp["fill"][:, :, p]
    return {f"{prefix}_{name}": values for name, values in features.items()}


### Feature group `relaxed_lp_best`

Value terminal water with the efficient production segment, not full-load average.

In [ ]:
import numpy as np

def build_relaxed_lp_best(data):
    lp = solve_all(data, np.arange(data.n_cases), workers=1, min_flow=True, commitment=True,
                   production_curves=True, flow_costs='yaml', main_river=True,
                   terminal_efficiency='best')
    return lp_features(data, lp, prefix='lpb')


### Feature group `lp_dispatch_context`

Horizon context and start-cost regret of the LP's effective hourly margins.

In [ ]:
CACHE = False  # cheap to derive; avoid a large redundant disk cache

import numpy as np
from scipy.ndimage import uniform_filter1d

def build_lp_dispatch_context(data):
    a, names = load_group('relaxed_lp_best', data)
    get = lambda n: np.asarray(a[..., names.index('lpb_'+n)])
    prod, on, margin = get('production'), get('unit_on'), get('margin')
    out = {}
    for n, x in [('on',on), ('margin',margin)]:
        out[f'ldc_{n}_mean6'] = uniform_filter1d(x,6,axis=2,mode='nearest')
        out[f'ldc_{n}_mean24'] = uniform_filter1d(x,24,axis=2,mode='nearest')
        out[f'ldc_{n}_week'] = x.mean(axis=2,keepdims=True)
    out['ldc_production_week'] = prod.mean(axis=2,keepdims=True)
    out['ldc_on_min'] = on.min(axis=2,keepdims=True)
    out['ldc_on_max'] = on.max(axis=2,keepdims=True)
    out['ldc_margin_min'] = margin.min(axis=2,keepdims=True)
    out['ldc_margin_max'] = margin.max(axis=2,keepdims=True)
    out['ldc_margin_positive'] = (margin > 1e-4).mean(axis=2,keepdims=True)
    for plant in ['Tokke','Vinje']:
        u = next(i for i,n in enumerate(data.units) if plant_of(n)==plant)
        out[f'ldc_{plant}_production'] = prod[:,u:u+1]
        out[f'ldc_{plant}_margin'] = margin[:,u:u+1]
    scale = np.abs(week_prices(data)).mean(axis=1,keepdims=True)+1
    regret = np.empty_like(margin)
    for u, unit in enumerate(data.units):
        g = data.generators[unit]
        m = margin[:,u]*scale
        # At a negative margin an ON unit chooses minimum output.
        reward = m*np.where(m>=0,g['p_max'],g['p_min'])
        values = _path_values(np.stack([np.zeros_like(reward),reward],axis=2),
                   g['startcost_const']*np.array([[0,1],[1,0]]))
        regret[:,u] = np.clip((values[...,1]-values[...,0])/(scale*g['p_max']),-50,50)
    out['ldc_margin_regret'] = regret
    return out


## 4. Feature matrix

In [ ]:
FEATURE_GROUPS = ['price_shape', 'calendar', 'water_value', 'hydro', 'unit_static', 'wv_margin', 'scarcity', 'case_economics', 'relaxed_dp', 'tokke_vinje', 'relaxed_lp_best', 'lp_dispatch_context']
DROP_FEATURES = ['price', 'price_wkmean', 'wv_up_end', 'wv_up_start', 'wv_dn_end', 'net_wv']  # raw EUR levels: regime-dependent (2022 vs 2023–24)


def build_matrix(data):
    """All groups side by side, shape (case, unit, hour, feature), float32."""
    metadata, names = [], []
    for group in FEATURE_GROUPS:
        array, group_names = load_group(group, data)
        keep = [j for j, name in enumerate(group_names) if name not in DROP_FEATURES]
        metadata.append((array, keep))
        names += [group_names[j] for j in keep]
    X = np.empty((data.n_cases, len(data.units), HOURS, len(names)), dtype=np.float32)
    column = 0
    for array, keep in metadata:
        for j in keep:
            X[..., column] = array[..., j]
            column += 1
    return X, names


data = load_data()
X, names = build_matrix(data)
_groups.clear()  # release cached groups before model training
print(X.shape, len(names), "features")

## 5. Model

### Model wrapper

Model training on the long format (one row per case × unit × hour).

Model types (`model: type:` in a config): `lightgbm`, `xgboost`, `random_forest`,
`logistic`, `mlp` (the last three to compare predictor types; report §5), or `blend` (a weighted
average of the raw probabilities of several member models, each with its own model
config under `members:`; global weights, never per unit, BRIEFING §4.2). Add new types to
`_train`, `_predict_one` and `_gain`; the rest of the harness doesn't need to change.
With `per_unit: true`, one model is trained per unit (a list of 14 models) instead of one
pooled model with the unit as a feature.

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler



N_JOBS = 4  # the grading PC has at most 4 cores; a fixed thread count keeps results reproducible


def case_weights(model_cfg: dict) -> np.ndarray | None:
    """Training weight per labelled case, or None for equal weights.

    `slow_solver: {seconds: s, weight: w}` gives cases whose SHOP run took ≥ s seconds the
    weight w: slow runs cluster near a time limit and may have suboptimal labels (BRIEFING
    §2.3). total_calculation_time is used only as a training weight, never as a feature.
    """
    slow = model_cfg.get("slow_solver")
    if slow is None:
        return None
    seconds = pd.read_csv(KERNEL_DIR / "Unit_commitment_decisions.csv",
                          usecols=["total_calculation_time"])["total_calculation_time"].to_numpy()
    return np.where(seconds >= slow["seconds"], slow["weight"], 1.0)


def _train(model_cfg: dict, X: np.ndarray, y: np.ndarray, names: list[str], categorical: list[str],
           weight: np.ndarray | None = None):
    seed = model_cfg.get("seed", 0)
    if model_cfg["type"] == "lightgbm":
        # Fixed seed and thread count plus deterministic=True / force_row_wise=True make two
        # runs give byte-identical predictions (needed for the short notebooks)
        params = {"random_state": seed, "n_jobs": N_JOBS, "deterministic": True,
                  "force_row_wise": True, **model_cfg["params"]}
        model = lgb.LGBMClassifier(verbose=-1, **params)
        model.fit(X, y, sample_weight=weight, feature_name=names, categorical_feature=categorical)
        return model
    if model_cfg["type"] == "xgboost":
        # The unit ID is used as an ordinary numeric feature here (no categorical support
        # for plain numpy input); trees can still split on it.
        params = {"random_state": seed, "n_jobs": N_JOBS, **model_cfg["params"]}
        model = xgb.XGBClassifier(tree_method="hist", importance_type="gain", **params)
        model.fit(X, y, sample_weight=weight)
        return model
    if model_cfg["type"] == "random_forest":
        # Bagged trees (no boosting); sklearn handles the coupling features' NaN natively.
        # The unit ID is numeric, as for XGBoost.
        params = {"random_state": seed, "n_jobs": N_JOBS, **model_cfg["params"]}
        model = RandomForestClassifier(**params)
        model.fit(X, y, sample_weight=weight)
        return model
    if model_cfg["type"] in ("logistic", "mlp"):
        # Linear model and neural network need scaled inputs without NaN: the unit ID is
        # one-hot encoded, missing values (partner features of non-Tokke/Vinje units) become
        # 0 plus a missing indicator, and every column is standardised.
        unit = [names.index(c) for c in categorical]
        rest = [j for j in range(X.shape[1]) if j not in unit]
        prep = ColumnTransformer([
            ("unit", OneHotEncoder(handle_unknown="ignore"), unit),
            ("num", make_pipeline(SimpleImputer(strategy="constant", fill_value=0, add_indicator=True),
                                  StandardScaler()), rest)])
        if model_cfg["type"] == "logistic":
            estimator = LogisticRegression(**model_cfg["params"])
        else:
            estimator = MLPClassifier(random_state=seed, **model_cfg["params"])
        model = Pipeline([("prep", prep), ("model", estimator)])
        model.fit(X, y)
        return model
    raise ValueError(f"unknown model type: {model_cfg['type']}")


def _predict_one(model, X: np.ndarray) -> np.ndarray:
    if isinstance(model, lgb.LGBMClassifier):
        return model.booster_.predict(X)
    return model.predict_proba(X)[:, 1]


def _gain(model) -> np.ndarray:
    if isinstance(model, lgb.LGBMClassifier):
        gain = model.booster_.feature_importance("gain")
    elif isinstance(model, Pipeline):  # logistic / MLP: no split gain
        return np.full(model.n_features_in_, np.nan)
    else:
        gain = model.feature_importances_
    return gain / max(gain.sum(), 1e-12)


class Blend:
    """Weighted average of member models' probabilities (weights normalised to sum 1)."""

    def __init__(self, members: list[tuple[float, object]]):
        total = sum(w for w, _ in members)
        self.members = [(w / total, m) for w, m in members]


def fit(model_cfg: dict, X: np.ndarray, Y: np.ndarray, case_idx: np.ndarray, names: list[str]):
    """Train on the given cases. X is (case, unit, hour, feature), Y is (case, unit, hour).

    Rows are subsampled to `row_share` of the training rows to fit in memory.
    """
    if model_cfg["type"] == "blend":
        return Blend([(m["weight"], fit(m, X, Y, case_idx, names)) for m in model_cfg["members"]])
    rng = np.random.default_rng(model_cfg.get("seed", 0))
    n_features = X.shape[-1]

    if model_cfg.get("per_unit"):
        models = []
        for u in range(X.shape[1]):
            X_unit = X[case_idx, u].reshape(-1, n_features)
            y_unit = Y[case_idx, u].ravel()
            keep = rng.random(len(y_unit)) < model_cfg["row_share"]
            models.append(_train(model_cfg, X_unit[keep], y_unit[keep], names, categorical=[]))
        return models

    y_train = Y[case_idx].ravel()
    keep = rng.random(len(y_train)) < model_cfg["row_share"]
    rows_per_case = X.shape[1] * X.shape[2]
    selected = np.flatnonzero(keep)
    absolute = case_idx[selected // rows_per_case] * rows_per_case + selected % rows_per_case
    X_train = X.reshape(-1, n_features)[absolute]
    categorical = ["unit"] if "unit" in names else []
    weights = case_weights(model_cfg)
    if weights is not None:
        weights = np.broadcast_to(weights[case_idx, None, None], Y[case_idx].shape).ravel()[keep]
    return _train(model_cfg, X_train, y_train[keep], names, categorical, weights)


def predict(model, X: np.ndarray, case_idx: np.ndarray) -> np.ndarray:
    """P(ON) for the given cases, shape (case, unit, hour)."""
    if isinstance(model, Blend):
        return sum(w * predict(m, X, case_idx) for w, m in model.members)
    X_cases = X[case_idx]
    if isinstance(model, list):  # one model per unit
        return np.stack([_predict_one(m, X_cases[:, u].reshape(-1, X.shape[-1])).reshape(len(case_idx), -1)
                         for u, m in enumerate(model)], axis=1)
    return _predict_one(model, X_cases.reshape(-1, X.shape[-1])).reshape(X_cases.shape[:-1])


def importance(model, names: list[str]) -> pd.Series:
    """Share of the total split gain per feature (averaged over units for per-unit models)."""
    if isinstance(model, Blend):
        return sum(w * importance(m, names) for w, m in model.members)
    models = model if isinstance(model, list) else [model]
    return pd.Series(np.mean([_gain(m) for m in models], axis=0), index=names)


## 6. Train on all 2015–2022 cases and write the submission

Settings below reproduce the selected experiment; its validation mean is recorded in the introduction.

In [ ]:
MODEL = {'type': 'xgboost', 'row_share': 0.6, 'seed': 0, 'params': {'n_estimators': 2200, 'learning_rate': 0.03, 'grow_policy': 'lossguide', 'max_leaves': 127, 'max_depth': 0, 'min_child_weight': 50, 'subsample': 0.8, 'colsample_bytree': 0.8}}

train_idx = np.arange(data.n_train)
test_idx = np.arange(data.n_train, data.n_cases)
model = fit(MODEL, X, data.Y, train_idx, names)
pred = predict(model, X, test_idx)

submission = pd.DataFrame(pred.reshape(len(test_idx), -1), columns=data.label_columns)
submission.insert(0, "Run No", data.run_no[test_idx])

# Format checks against sample_submission.csv
sample = pd.read_csv(ROOT / "sample_submission.csv")
assert list(submission.columns) == list(sample.columns)
assert (submission["Run No"].to_numpy() == sample["Run No"].to_numpy()).all()
values = submission.iloc[:, 1:].to_numpy()
assert np.isfinite(values).all() and values.min() >= 0 and values.max() <= 1

submission.to_csv(OUTPUT, index=False, float_format="%.6f")
print(f"written: {OUTPUT} ({len(submission)} rows)")